# Dịch full title Cado bằng Qwen3-14B-AWQ (4-bit AWQ) trên Google Colab

Notebook này chạy **`Qwen/Qwen3-14B-AWQ` (4-bit AWQ) trên GPU Colab** với sampling non-thinking chống lặp; không chạy Gemma, không gọi API dịch, không cần HF token và không tự upload kết quả lên Hugging Face.

- Nguồn được khóa theo revision và SHA-256 trong pipeline: 16.348 dòng, khoảng 15.863 title Trung duy nhất.
- Checkpoint Qwen riêng nằm trong `MyDrive/title_translation_cado_qwen3_14b_awq_full_v3_sampled`; không dùng chung checkpoint Gemma.
- Runtime bắt đầu với batch 1; không tăng batch trước khi đo tốc độ và VRAM trên runtime thật. Cache SQLite được sao lưu lên Drive mỗi 100 title đã xử lý.
- Notebook chạy preflight 20 title. Nếu còn title chưa hoàn tất hoặc đầu ra lặp/cắt cụt, notebook dừng trước full và vẫn giữ checkpoint; các cờ QA kỹ thuật còn lại có thể được rà soát hậu kỳ theo lựa chọn của bạn.
- Nếu Colab ngắt, kết nối lại cùng loại runtime và chọn **Runtime → Run all**. Các bản dịch đã checkpoint sẽ được bỏ qua; model weights có thể cần tải lại.

`semantic_verified` luôn là `False`: QA tự động không thay thế kiểm tra ngữ nghĩa của con người. Notebook không tự publish lên HF.


Bản v6 tạo run folder riêng. Cache v2 không bị sửa; chỉ các bản hoàn chỉnh, không bị lặp mới được nhập, còn lỗi/lặp sẽ chạy lại theo decoder mới.


## Trước khi chạy

1. Mở notebook trong Google Colab.
2. Chọn **Runtime → Change runtime type → GPU**. T4 16 GB có thể chạy AWQ 4-bit nhưng khá sát giới hạn; bắt đầu batch 1, không bảo đảm mọi runtime T4 đều đủ VRAM.
3. Chọn **Runtime → Run all** và cấp quyền mount Google Drive khi Colab hỏi.
4. Bundle `title_translation_qwen3_14b_awq_colab_v6_sampled_autoreload_bundle.zip` nằm trong thư mục run trên Drive; notebook chỉ hỏi upload nếu file bị thiếu.

Không dán API key hay HF token vào notebook. Dataset và model đều được tải công khai theo revision đã khóa.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
WORKDIR = DRIVE_ROOT / 'title_translation_cado_qwen3_14b_awq_full_v3_sampled'
WORKDIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR = Path('/content/title_translation_cado_qwen3_14b_awq_full_v3_sampled')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE = LOCAL_DIR / 'cache_qwen3_14b_awq.sqlite3'
DRIVE_CACHE = WORKDIR / 'cache_qwen3_14b_awq.sqlite3'

if DRIVE_CACHE.exists() and not LOCAL_CACHE.exists():
    shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)
    print('Đã khôi phục SQLite checkpoint từ Drive:', DRIVE_CACHE)
else:
    print('Checkpoint SQLite local:', 'đã có' if LOCAL_CACHE.exists() else 'chưa có; run mới hoặc sẽ khôi phục khi có bản Drive')
print('Run folder riêng của Qwen:', WORKDIR)


# Ghi lỗi của mọi cell notebook lên Drive để có thể rà soát mà không cần chụp/paste traceback.
import re
import traceback
from datetime import datetime, timezone
from IPython import get_ipython
_qwen_status_path = WORKDIR / 'qwen_live_status.json'
def _qwen_redact(value):
    value = str(value)
    value = re.sub(r'hf_[A-Za-z0-9]{16,}', '[REDACTED_HF_TOKEN]', value)
    value = re.sub(r'AIza[0-9A-Za-z_-]{20,}', '[REDACTED_GOOGLE_KEY]', value)
    value = re.sub(r'AQ\.[A-Za-z0-9_-]{20,}', '[REDACTED_API_KEY]', value)
    return value
def _qwen_drive_error_hook(result):
    error = getattr(result, 'error_before_exec', None) or getattr(result, 'error_in_exec', None)
    if error is None:
        raw_cell = getattr(getattr(result, 'info', None), 'raw_cell', '') or ''
        if '_qwen_full_run_cell_marker' not in raw_cell:
            return
        summary = globals().get('full_result')
        if not isinstance(summary, dict):
            return
        try:
            payload = {
                'state': summary.get('status', 'unknown'),
                'phase': 'run_cell_finished',
                'updated_at_utc': datetime.now(timezone.utc).isoformat(),
                'completed_unique_titles': summary.get('completed_unique_titles'),
                'total_unique_titles': summary.get('total_unique_titles'),
                'remaining_unique_titles': summary.get('remaining_unique_titles'),
                'generation': summary.get('generation'),
            }
            tmp = _qwen_status_path.with_suffix('.json.tmp')
            tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
            tmp.replace(_qwen_status_path)
        except Exception as log_error:
            print(f'[qwen-drive-observer] could not write final status: {type(log_error).__name__}: {log_error}', flush=True)
        return
    try:
        payload = {
            'state': 'error',
            'phase': 'notebook_cell_error',
            'updated_at_utc': datetime.now(timezone.utc).isoformat(),
            'cell_execution_count': getattr(result, 'execution_count', None),
            'last_error_class': type(error).__name__,
        }
        tmp = _qwen_status_path.with_suffix('.json.tmp')
        tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
        tmp.replace(_qwen_status_path)
    except Exception as log_error:
        print(f'[qwen-drive-observer] could not write cell error: {type(log_error).__name__}: {log_error}', flush=True)
_qwen_ip = get_ipython()
if _qwen_ip is not None:
    _old_hook = getattr(_qwen_ip, '_qwen_drive_error_hook', None)
    if _old_hook is not None:
        try: _qwen_ip.events.unregister('post_run_cell', _old_hook)
        except Exception: pass
    _qwen_ip.events.register('post_run_cell', _qwen_drive_error_hook)
    _qwen_ip._qwen_drive_error_hook = _qwen_drive_error_hook


In [ ]:
# Cài thư viện Transformers cho AWQ; không nâng cấp PyTorch/CUDA có sẵn trong Colab.
import importlib
import importlib.metadata
import importlib.util
import json
import subprocess
import sys
from packaging.version import Version

PACKAGE_LOCK = WORKDIR / 'qwen3_14b_awq_colab_v2_package_lock.json'
FAST_TRANSFORMERS_VERSION = '5.20.2'
MIN_SUPPORTED_TRANSFORMERS = Version('5.17.0')
PACKAGE_NAMES = ['transformers', 'accelerate', 'gptqmodel', 'huggingface_hub', 'sacrebleu', 'pyarrow', 'pandas', 'numpy']
PACKAGE_SPECS = [
    f'transformers=={FAST_TRANSFORMERS_VERSION}', 'accelerate>=1.2', 'gptqmodel',
    'huggingface_hub>=0.30', 'sacrebleu', 'pyarrow>=18', 'pandas>=2.2,<3',
]

def installed_package_versions():
    versions = {}
    for name in PACKAGE_NAMES:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            # A persisted lock can outlive a Colab runtime reset; represent absent
            # packages as mismatches so the locked versions are installed below.
            versions[name] = None
    versions['torch'] = Version(importlib.metadata.version('torch')).base_version  # ignore +cpu/+cuXXX build tag
    versions['numpy'] = importlib.metadata.version('numpy')  # preserve Colab ABI
    return versions

def _pip_install(command):
    from collections import deque
    tail = deque(maxlen=300)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        tail.append(line)
    return_code = process.wait()
    log_text = ''.join(tail)
    redactor = globals().get('_qwen_redact')
    if callable(redactor):
        log_text = redactor(log_text)
    (WORKDIR / 'qwen_install_last.log').write_text(log_text, encoding='utf-8')
    if return_code:
        raise RuntimeError(f'pip install exited with code {return_code}; last output follows:\n{log_text[-10000:]}')

if PACKAGE_LOCK.exists():
    locked_packages = json.loads(PACKAGE_LOCK.read_text(encoding='utf-8'))
    if locked_packages.get('torch'):
        locked_packages['torch'] = Version(locked_packages['torch']).base_version
    locked_transformers = Version(locked_packages['transformers'])
    if locked_transformers < MIN_SUPPORTED_TRANSFORMERS:
        print(
            f'Lock Transformers cũ ({locked_transformers}); nâng lên {FAST_TRANSFORMERS_VERSION} '
            'để dùng phiên bản Transformers đã khóa cho Qwen3 AWQ.'
        )
        locked_packages['transformers'] = FAST_TRANSFORMERS_VERSION
    current_packages = installed_package_versions()
    if current_packages != locked_packages:
        exact_specs = [f'{name}=={locked_packages[name]}' for name in PACKAGE_NAMES]
        torch_constraint = Path('/tmp/qwen3_14b_awq_torch_constraint.txt')
        torch_constraint.write_text(f"torch=={importlib.metadata.version('torch')}\n"f"numpy=={importlib.metadata.version('numpy')}\n", encoding='utf-8')
        _pip_install([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--constraint', str(torch_constraint), *exact_specs])
        importlib.invalidate_caches()
        current_packages = installed_package_versions()
        if current_packages != locked_packages:
            raise RuntimeError(f'Không khôi phục được môi trường đã khóa. Mong đợi {locked_packages}; hiện có {current_packages}.')
        tmp_lock = PACKAGE_LOCK.with_suffix('.json.tmp')
        tmp_lock.write_text(json.dumps(locked_packages, indent=2), encoding='utf-8')
        tmp_lock.replace(PACKAGE_LOCK)
else:
    torch_constraint = Path('/tmp/qwen3_14b_awq_torch_constraint.txt')
    torch_constraint.write_text(f"torch=={importlib.metadata.version('torch')}\n"f"numpy=={importlib.metadata.version('numpy')}\n", encoding='utf-8')
    _pip_install([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--constraint', str(torch_constraint), *PACKAGE_SPECS])
    importlib.invalidate_caches()
    locked_packages = installed_package_versions()
    tmp_lock = PACKAGE_LOCK.with_suffix('.json.tmp')
    tmp_lock.write_text(json.dumps(locked_packages, indent=2), encoding='utf-8')
    tmp_lock.replace(PACKAGE_LOCK)

tmp_lock = PACKAGE_LOCK.with_suffix('.json.normalized.tmp')
tmp_lock.write_text(json.dumps(locked_packages, indent=2), encoding='utf-8')
tmp_lock.replace(PACKAGE_LOCK)
from transformers.utils.import_utils import is_gptqmodel_available
if not is_gptqmodel_available():
    raise RuntimeError('gptqmodel chưa được cài/nhận diện; xem lỗi cài package ở cell này rồi chạy lại.')
print('gptqmodel:', importlib.metadata.version('gptqmodel'))
print('Môi trường khóa trên Drive:', json.dumps(locked_packages, indent=2))


In [ ]:
# Nạp mã pipeline hiện tại từ bundle và tải đúng Parquet nguồn công khai.
import hashlib
import importlib
import io
import json
import sys
import zipfile
from huggingface_hub import hf_hub_download

BUNDLE_PATH = WORKDIR / 'title_translation_qwen3_14b_awq_colab_v6_sampled_autoreload_bundle.zip'
if not BUNDLE_PATH.exists():
    print('Upload file title_translation_qwen3_14b_awq_colab_v6_sampled_autoreload_bundle.zip')
    uploaded = files.upload()
    if 'title_translation_qwen3_14b_awq_colab_v6_sampled_autoreload_bundle.zip' not in uploaded:
        raise FileNotFoundError('Cần upload title_translation_qwen3_14b_awq_colab_v6_sampled_autoreload_bundle.zip')
    BUNDLE_PATH.write_bytes(uploaded['title_translation_qwen3_14b_awq_colab_v6_sampled_autoreload_bundle.zip'])

PROJECT_DIR = Path('/content/title_translation_qwen3_14b_awq_v6_sampled_source')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MAX_BUNDLE_BYTES = 10 * 1024 * 1024
EXPECTED_BUNDLE_FILES = {'title_translation_hybrid.py', 'title_translation_pipeline.py'}

def load_pipeline_bundle(expected_sha256=None):
    bundle_bytes = BUNDLE_PATH.read_bytes()
    if len(bundle_bytes) > MAX_BUNDLE_BYTES:
        raise RuntimeError(f'Bundle lớn bất thường: {len(bundle_bytes)} bytes.')
    bundle_sha256 = hashlib.sha256(bundle_bytes).hexdigest()
    if expected_sha256 and bundle_sha256.lower() != str(expected_sha256).lower():
        raise RuntimeError(f'SHA-256 bundle không khớp lệnh reload: {bundle_sha256}')
    with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as zf:
        names = set(zf.namelist())
        if names != EXPECTED_BUNDLE_FILES:
            raise RuntimeError(f'Bundle có tên tệp không được phép: {sorted(names)}')
        for name in EXPECTED_BUNDLE_FILES:
            info = zf.getinfo(name)
            if info.file_size > 2 * 1024 * 1024:
                raise RuntimeError(f'Module trong bundle lớn bất thường: {name}')
            (PROJECT_DIR / name).write_bytes(zf.read(name))
    sys.path.insert(0, str(PROJECT_DIR)) if str(PROJECT_DIR) not in sys.path else None
    importlib.invalidate_caches()
    sys.modules.pop('title_translation_hybrid', None)
    sys.modules.pop('title_translation_pipeline', None)
    module = importlib.import_module('title_translation_hybrid')
    return module, bundle_sha256

pipeline, ACTIVE_BUNDLE_SHA = load_pipeline_bundle()
print('Đã nạp pipeline từ bundle Drive; SHA-256:', ACTIVE_BUNDLE_SHA)

SOURCE = WORKDIR / Path(pipeline.DATASET_FILE).name
if not SOURCE.exists():
    downloaded = hf_hub_download(
        repo_id=pipeline.DATASET_ID,
        filename=pipeline.DATASET_FILE,
        repo_type='dataset',
        revision=pipeline.DATASET_REVISION,
    )
    shutil.copy2(downloaded, SOURCE)
actual_sha = pipeline.quality.sha256_file(SOURCE)
if actual_sha != pipeline.DATASET_SHA256:
    raise ValueError(f'SHA-256 nguồn không khớp: {actual_sha}; dừng trước khi dịch.')
print(f'Nguồn đã xác minh: {pipeline.EXPECTED_ROWS:,} dòng; {pipeline.EXPECTED_UNIQUE_TITLES:,} title duy nhất; SHA-256={actual_sha}')


In [ ]:
# GPU check và hàm mirror SQLite từ SSD runtime sang Drive.
import json
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Chưa bật GPU. Chọn Runtime → Change runtime type → GPU rồi chạy lại.')
gpu = torch.cuda.get_device_properties(0)
gpu_vram_gib = gpu.total_memory / (1024 ** 3)
print(f'GPU: {gpu.name}; VRAM: {gpu_vram_gib:.1f} GiB')
if gpu.total_memory < 14 * 1024 ** 3:
    raise RuntimeError('Qwen3-14B AWQ 4-bit cần tối thiểu 14 GiB VRAM. Hãy chọn runtime GPU phù hợp.')

def persist_qwen_cache(source_path):
    source_path = Path(source_path)
    if not source_path.exists():
        return
    temporary = DRIVE_CACHE.with_name(DRIVE_CACHE.name + '.uploading')
    shutil.copy2(source_path, temporary)
    temporary.replace(DRIVE_CACHE)

print('Cache sẽ được đồng bộ lên Drive định kỳ; cache local:', LOCAL_CACHE)


## Chạy full Qwen

Chạy cell dưới đây. Preflight chỉ kiểm tra model/runtime và chuẩn bị checkpoint; kết quả QA của 20 title không dùng làm điều kiện chặn full. Toàn bộ title duy nhất sẽ được dịch trong một lượt dài, với SQLite checkpoint định kỳ. Nếu runtime bị ngắt, chạy lại notebook từ đầu để tiếp tục từ bản checkpoint gần nhất trên Drive.

Không đổi model, prompt hoặc bundle giữa chừng. Nếu đổi cấu hình, hãy tạo run folder mới.


### Theo dõi và can thiệp qua Drive

Trong thư mục run, `qwen_live_status.json` ghi tiến độ/lỗi gần nhất; `qwen_control.json` nhận `run`, `pause`, `stop`, `reload`. Sau khi upload bundle mới, đặt lệnh `reload` kèm SHA-256; ở ranh giới batch, notebook lưu checkpoint, kiểm tra hash, nạp mã mới rồi tiếp tục. Lệnh chỉ là JSON allowlist, không thực thi Python.


In [ ]:
import time
from datetime import datetime, timezone
# Preflight 20 title, sau đó chạy full ngay cả khi QA preflight báo cờ.
_qwen_full_run_cell_marker = 'qwen-full-run-v6'
QWEN_BATCH_SIZE = 1  # Giữ batch đã khóa khi smoke/checkpoint được tạo
QWEN_FULL_BATCH_SIZE = 4  # Thử tối đa; pipeline tự hạ 4 -> 2 -> 1 nếu OOM lúc generate
CHECKPOINT_EVERY_TITLES = 100
CONTROL_FILE = WORKDIR / 'qwen_control.json'
LIVE_STATUS_FILE = WORKDIR / 'qwen_live_status.json'
CONTROL_POLL_SECONDS = 5
if not CONTROL_FILE.exists():
    CONTROL_FILE.write_text(json.dumps({'action': 'run'}, indent=2), encoding='utf-8')

_control_last_status_write = 0.0
_control_last_action = None
_control_expected_bundle_sha256 = None
_SAFE_CONTROL_ERROR_REASONS = {'repeated_phrase_guard', 'truncated_output', 'empty_output'}
_SAFE_EXCEPTION_CLASSES = {'RuntimeError', 'ImportError', 'ModuleNotFoundError', 'CalledProcessError', 'FileNotFoundError', 'TimeoutError', 'ConnectionError', 'HTTPError', 'ValueError', 'TypeError', 'KeyError', 'AttributeError', 'MemoryError', 'OSError'}

def _qwen_safe_recent_errors(items):
    safe_items = []
    for item in items if isinstance(items, list) else []:
        if not isinstance(item, dict):
            safe_items.append({'reason': 'other_error'})
            continue
        reason = str(item.get('reason') or '')
        safe_items.append({
            'reason': reason if reason in _SAFE_CONTROL_ERROR_REASONS else 'other_error',
            'truncated': bool(item.get('truncated')),
        })
    return safe_items[-3:]

def _qwen_safe_error_class(value):
    candidate = str(value or '').split(':', 1)[0].strip()
    return candidate if candidate in _SAFE_EXCEPTION_CLASSES else ('unknown_error' if value else None)

def qwen_drive_control(progress):
    global _control_last_status_write, _control_last_action, _control_expected_bundle_sha256
    command_error = None
    try:
        command = json.loads(CONTROL_FILE.read_text(encoding='utf-8'))
        action = str(command.get('action', 'run')).strip().lower()
        _control_expected_bundle_sha256 = command.get('bundle_sha256')
    except Exception as exc:
        action = _control_last_action or 'run'
        command = {'action': action}
        command_error = f'{type(exc).__name__}: {exc}'
    if action not in {'run', 'pause', 'stop', 'reload'}:
        command_error = f"Unsupported action {action!r}; expected run, pause, stop, or reload."
        action = 'run'
    now = time.time()
    processed = int(progress.get('processed_this_session', 0))
    should_write = (
        action != _control_last_action
        or processed % 20 == 0
        or now - _control_last_status_write >= 30
        or bool(progress.get('last_error'))
    )
    if should_write:
        status = {
            'updated_at_utc': datetime.now(timezone.utc).isoformat(),
            'model_key': progress.get('model_key'),
            'phase': progress.get('phase', 'unknown'),
            'action': action,
            'processed_this_session': processed,
            'total_this_session': int(progress.get('total_this_session', 0)),
            'completed_this_session': int(progress.get('completed_this_session', 0)),
            'errors_this_session': int(progress.get('errors_this_session', 0)),
            'completed_before_session': int(progress.get('completed_before_session', 0)),
            'completed_estimate': int(progress.get('completed_before_session', 0)) + int(progress.get('completed_this_session', 0)),
            'elapsed_seconds': float(progress.get('elapsed_seconds', 0) or 0),
            'throughput_titles_per_hour': float(progress.get('throughput_titles_per_hour', 0) or 0),
            'recent_errors': _qwen_safe_recent_errors(progress.get('recent_errors', [])),
            'last_error_class': _qwen_safe_error_class(progress.get('last_error')),
            'command_error': bool(command_error),
            'checkpoint_file': DRIVE_CACHE.name,
            'runtime': 'Colab; control polled between generation batches',
            'active_bundle_sha256': ACTIVE_BUNDLE_SHA,
            'requested_bundle_sha256': _control_expected_bundle_sha256,
        }
        try:
            temp_status = LIVE_STATUS_FILE.with_suffix('.json.tmp')
            temp_status.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
            temp_status.replace(LIVE_STATUS_FILE)
            _control_last_status_write = now
        except Exception as exc:
            print(f'[qwen-control] could not mirror status to Drive: {type(exc).__name__}: {exc}', flush=True)
    _control_last_action = action
    return action

def attach_durable_cache(context, enable_control=False):
    context['cache_paths'] = {'qwen3_14b_awq': LOCAL_CACHE}
    context['cache_checkpoint_interval'] = CHECKPOINT_EVERY_TITLES
    context['cache_checkpoint_callback'] = persist_qwen_cache
    if enable_control:
        context['control_callback'] = qwen_drive_control
        context['control_poll_seconds'] = CONTROL_POLL_SECONDS
    return context

def _apply_drive_code_update(timeout_seconds=600):
    global pipeline, ACTIVE_BUNDLE_SHA
    old_sha = ACTIVE_BUNDLE_SHA
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        try:
            candidate_sha = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
        except Exception as exc:
            candidate_sha = None
            print(f'[qwen-reload] cannot read Drive bundle: {type(exc).__name__}: {exc}', flush=True)
        expected_sha = _control_expected_bundle_sha256
        if candidate_sha and candidate_sha != old_sha and (not expected_sha or candidate_sha.lower() == str(expected_sha).lower()):
            new_pipeline, new_sha = load_pipeline_bundle(expected_sha256=expected_sha)
            pipeline = new_pipeline
            ACTIVE_BUNDLE_SHA = new_sha
            temp_control = CONTROL_FILE.with_suffix('.json.tmp')
            temp_control.write_text(json.dumps({'action': 'run', 'bundle_sha256': new_sha}, indent=2), encoding='utf-8')
            temp_control.replace(CONTROL_FILE)
            print(f'[qwen-reload] loaded reviewed bundle {new_sha}; resuming from SQLite checkpoint', flush=True)
            return True
        qwen_drive_control({'model_key': 'qwen3_14b_awq', 'phase': 'awaiting_code_update',
                            'processed_this_session': 0, 'total_this_session': 0,
                            'completed_this_session': 0, 'errors_this_session': 0,
                            'completed_before_session': 0})
        time.sleep(CONTROL_POLL_SECONDS)
    print('[qwen-reload] no new matching bundle arrived; checkpoint remains saved. Restart Run all after upload.', flush=True)
    return False

def _migrate_legacy_qwen_cache(context):
    """Copy only reusable completed rows; never mutate the v2 source checkpoint."""
    import sqlite3
    from datetime import datetime, timezone

    marker = WORKDIR / 'legacy_cache_import.json'
    if marker.exists():
        print('Đã hoàn tất nhập cache v2 trước đó; giữ nguyên cache v3.')
        return

    legacy_workdir = DRIVE_ROOT / 'title_translation_cado_qwen3_14b_awq_full_v2'
    legacy_cache = legacy_workdir / 'cache_qwen3_14b_awq.sqlite3'
    legacy_lock_path = legacy_workdir / 'hybrid_environment_lock.json'
    legacy_status_path = legacy_workdir / 'qwen_live_status.json'
    if legacy_status_path.exists():
        try:
            legacy_status = json.loads(legacy_status_path.read_text(encoding='utf-8'))
            last_state = str(legacy_status.get('state', '')).lower()
            updated = datetime.fromisoformat(str(legacy_status.get('updated_at_utc', '')).replace('Z', '+00:00'))
            is_recent = (datetime.now(timezone.utc) - updated).total_seconds() < 1200
            if is_recent and last_state in {'running', 'generating', 'loading_model', 'awaiting_code_update'}:
                raise RuntimeError('Run v2 có trạng thái hoạt động gần đây; hãy dừng hẳn run cũ trước khi nhập cache.')
        except ValueError:
            pass

    if not legacy_cache.exists() or not legacy_lock_path.exists():
        print('Không tìm thấy đủ cache/lock v2; v3 sẽ tiếp tục bằng cache riêng mới.')
        marker.write_text(json.dumps({'status': 'no_legacy_cache'}, indent=2), encoding='utf-8')
        return

    legacy_lock = json.loads(legacy_lock_path.read_text(encoding='utf-8'))
    model_key = 'qwen3_14b_awq'
    old_model = legacy_lock.get('models', {}).get(model_key, {})
    old_revision = old_model.get('revision')
    if old_model.get('model_id') != pipeline.MODELS[model_key]['model_id']:
        raise RuntimeError('Cache v2 thuộc model khác; dừng để tránh trộn kết quả.')

    # Pin v3 to the immutable revision used by the old cache before computing
    # the new config ID. This allows safe reuse while recording the new decoder.
    if old_revision:
        new_lock = context['lock']
        new_models = dict(new_lock['models'])
        new_model = dict(new_models[model_key])
        new_model['revision'] = old_revision
        new_models[model_key] = new_model
        new_config = pipeline._config_payload(
            new_models,
            qwen_batch_size=int(context.get('qwen_batch_size', 1)),
        )
        new_config_id = pipeline.fingerprint(new_config)
        new_lock.update({'models': new_models, 'config': new_config, 'config_id': new_config_id})
        context['lock'] = new_lock
        context['state']['config_id'] = new_config_id
        pipeline.atomic_json(WORKDIR / 'hybrid_environment_lock.json', new_lock)
        pipeline.atomic_json(WORKDIR / 'hybrid_run_state.json', context['state'])

    revision = context['lock']['models'][model_key]['revision']
    target_cache = pipeline.Cache(LOCAL_CACHE, model_key, revision, context['lock']['config_id'])
    imported = 0
    skipped = 0
    skipped_smoke = 0
    smoke_titles = set(context.get('samples', {}).get('smoke', []))
    with sqlite3.connect(f'file:{legacy_cache.as_posix()}?mode=ro', uri=True) as old_db:
        rows = old_db.execute(
            "SELECT source_title,output_text,status,truncated,input_tokens,output_tokens,attempts,"
            "api_requests,elapsed_seconds,error_text,model_version,response_id "
            "FROM translations"
        ).fetchall()
    for row in rows:
        title, output, status, truncated, in_tokens, out_tokens, attempts, api_requests, elapsed, error, version, response_id = row
        if str(title) in smoke_titles:
            # Force the smoke set through the new sampling decoder so the
            # preflight actually tests v6 instead of passing on legacy cache.
            skipped_smoke += 1
            continue
        record = {
            'output_text': output,
            'status': status,
            'truncated': bool(truncated),
            'input_tokens': in_tokens,
            'output_tokens': out_tokens,
            'attempts': attempts or 0,
            'api_requests': api_requests or attempts or 0,
            'elapsed_seconds': elapsed or 0.0,
            'error_text': error,
            'model_version': (version or 'legacy-qwen3-14b-awq') + '; legacy-greedy',
            'response_id': response_id,
            'generation_prompt_version': 'legacy-qwen3-14b-awq-greedy-v1',
        }
        if not pipeline._complete(record):
            skipped += 1
            continue
        target_cache.put(str(title), record)
        imported += 1

    persist_qwen_cache(LOCAL_CACHE)
    payload = {
        'status': 'complete',
        'source_run_dir': 'title_translation_cado_qwen3_14b_awq_full_v2',
        'source_cache_rows': len(rows),
        'imported_completed_nonrepetitive': imported,
        'skipped_incomplete_or_repetitive': skipped,
        'skipped_smoke_for_fresh_sampling_test': skipped_smoke,
        'source_revision': old_revision,
        'target_config_id': context['lock']['config_id'],
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    temp_marker = marker.with_suffix('.json.tmp')
    temp_marker.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    temp_marker.replace(marker)
    print('Nhập cache hoàn tất:', json.dumps(payload, ensure_ascii=False))

smoke_context = pipeline.prepare_pipeline(
    mode='smoke', model_key='qwen3_14b_awq', data_path=SOURCE, workdir=WORKDIR,
    hf_token=None, qwen_batch_size=QWEN_BATCH_SIZE
)
attach_durable_cache(smoke_context, enable_control=True)
_migrate_legacy_qwen_cache(smoke_context)
smoke_result = pipeline.run_stage(
    smoke_context, batch_size=QWEN_BATCH_SIZE, allow_qwen_hardware_change=True
)
_smoke_reload_count = 0
while smoke_result.get('status') == 'reload_requested' and _smoke_reload_count < 5:
    _smoke_reload_count += 1
    if not _apply_drive_code_update():
        break
    smoke_context = pipeline.prepare_pipeline(mode='smoke', model_key='qwen3_14b_awq', data_path=SOURCE, workdir=WORKDIR, hf_token=None, qwen_batch_size=QWEN_BATCH_SIZE)
    attach_durable_cache(smoke_context, enable_control=True)
    smoke_result = pipeline.run_stage(smoke_context, batch_size=QWEN_BATCH_SIZE, allow_qwen_hardware_change=True)
persist_qwen_cache(LOCAL_CACHE)
import gc
import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f'VRAM trống sau smoke: {free_bytes / 1024**3:.2f}/{total_bytes / 1024**3:.2f} GiB')
print('Preflight:', json.dumps({k: v for k, v in smoke_result.items() if k not in {'qa', 'cache'}}, ensure_ascii=False, indent=2, default=str))
smoke_completion = float(smoke_result.get('completion_rate', 0) or 0)
if smoke_completion < 0.99:
    raise RuntimeError(
        f'Preflight chỉ hoàn tất {smoke_completion:.1%}; đã lưu checkpoint lên Drive và dừng trước full. '
        'Không chạy full với đầu ra đang lỗi/lặp. Hãy dùng bundle đã sửa.'
    )
if smoke_result.get('status') == 'quality_gate_failed':
    print('Preflight đủ đầu ra nhưng còn cờ QA kỹ thuật; theo lựa chọn của bạn, sẽ lưu cờ để rà soát sau full.')
smoke_generation = smoke_result.get('generation', {})
smoke_new = int(smoke_generation.get('new_completed', 0))
smoke_seconds = float(smoke_generation.get('elapsed_seconds', 0) or 0)
if smoke_new and smoke_seconds > 0:
    rate = smoke_new / smoke_seconds
    eta_hours = pipeline.EXPECTED_UNIQUE_TITLES / rate / 3600
    print(f'Tốc độ preflight: {rate * 3600:.1f} title/giờ; ETA thô cho full: {eta_hours:.1f} giờ (chưa tính reset/tải model lại).')

full_result = None
for resume_round in range(1, 4):
    full_context = pipeline.prepare_pipeline(
        mode='full', model_key='qwen3_14b_awq', data_path=SOURCE, workdir=WORKDIR,
        hf_token=None, qwen_batch_size=QWEN_BATCH_SIZE
    )
    attach_durable_cache(full_context, enable_control=True)
    full_result = pipeline.run_stage(
        full_context,
        batch_size=QWEN_FULL_BATCH_SIZE,
        allow_qwen_batch_change=True,
        full_titles_per_session=pipeline.EXPECTED_UNIQUE_TITLES,
        allow_unvalidated_full=True,
        defer_full_quality_review=True,
        allow_qwen_hardware_change=True,
    )
    persist_qwen_cache(LOCAL_CACHE)
    if full_result.get('status') == 'reload_requested':
        if _apply_drive_code_update():
            continue
        break
    if full_result.get('status') != 'checkpoint_saved_continue':
        break
    if resume_round < 3:
        print(f"Còn title chưa hoàn tất; sẽ thử lại lượt {resume_round + 1}/3 từ checkpoint.")
    else:
        print('Vẫn còn title chưa hoàn tất sau 3 lượt; checkpoint còn trên Drive. Chạy lại notebook để thử tiếp.')
assert full_result is not None
print('Full run:', json.dumps({k: v for k, v in full_result.items() if k not in {'qa', 'cache'}}, ensure_ascii=False, indent=2, default=str))
if full_result.get('status') != 'all_titles_complete':
    print('Chưa xong toàn bộ; checkpoint đã lưu trên Drive. Sau khi nối runtime lại, chọn Runtime → Run all để resume.')


## Rà soát sau khi dịch xong

Chỉ chạy cell này sau khi full run có trạng thái `all_titles_complete`. Cell tạo CSV toàn bộ title, CSV các title cần xem lại và báo cáo tổng hợp. Cờ QA là kiểm tra kỹ thuật (số, mã, ký tự/ngôn ngữ, đầu ra), không chứng minh bản dịch đúng nghĩa.


In [ ]:
import json
import hashlib
import pandas as pd

state_path = WORKDIR / 'hybrid_run_state.json'
state = json.loads(state_path.read_text(encoding='utf-8'))
full_status = state.get('models', {}).get('qwen3_14b_awq', {}).get('stages', {}).get('full', {}).get('status')
if full_status != 'all_titles_complete':
    raise RuntimeError(f'Qwen full chưa hoàn tất (status={full_status}); resume full trước khi rà soát.')

OUTPUT = WORKDIR / 'bilingual_zh_vi_cado_title_qwen3_14b_awq.parquet'
if not OUTPUT.exists():
    raise FileNotFoundError(f'Không thấy Parquet hoàn chỉnh: {OUTPUT}')
result_frame = pd.read_parquet(OUTPUT)
if len(result_frame) != pipeline.EXPECTED_ROWS:
    raise AssertionError(f'Số dòng sai: {len(result_frame)}; cần {pipeline.EXPECTED_ROWS}.')
qa_status_col = 'title_vi_qwen3_14b_awq_qa_status'
hard_flags_col = 'title_vi_qwen3_14b_awq_qa_flags'
advisory_col = 'title_vi_qwen3_14b_awq_advisory_flags'
semantic_col = 'title_vi_qwen3_14b_awq_semantic_verified'
required = [qa_status_col, hard_flags_col, advisory_col, semantic_col, 'title_vi_candidate_qwen3_14b_awq']
missing = [name for name in required if name not in result_frame.columns]
if missing:
    raise AssertionError(f'Thiếu cột QA/output: {missing}')

unique_review = result_frame.drop_duplicates('title_zh').copy()
review_cols = [name for name in [
    'product_id', 'title_zh', 'title_vi', 'title_vi_cleaned',
    'title_vi_candidate_qwen3_14b_awq', qa_status_col, hard_flags_col, advisory_col,
] if name in unique_review.columns]
all_review_path = WORKDIR / 'qwen3_14b_awq_postrun_qa_all_unique_titles.csv'
unique_review[review_cols].to_csv(all_review_path, index=False, encoding='utf-8-sig')
flagged = unique_review[
    unique_review[qa_status_col].ne('pass')
    | unique_review[hard_flags_col].fillna('[]').ne('[]')
    | unique_review[advisory_col].fillna('[]').ne('[]')
]
flagged_path = WORKDIR / 'qwen3_14b_awq_postrun_qa_flagged_titles.csv'
flagged[review_cols].to_csv(flagged_path, index=False, encoding='utf-8-sig')

status_counts = unique_review[qa_status_col].fillna('missing').value_counts().to_dict()
hard_counts = {}
for value in unique_review[hard_flags_col].fillna('[]'):
    for flag in json.loads(value):
        hard_counts[flag] = hard_counts.get(flag, 0) + 1
summary = {
    'rows': len(result_frame),
    'unique_titles': int(unique_review['title_zh'].nunique()),
    'qa_status_counts_unique_titles': status_counts,
    'hard_flag_counts_unique_titles': hard_counts,
    'flagged_unique_titles': len(flagged),
    'semantic_verified': bool(result_frame[semantic_col].fillna(True).astype(bool).any()),
    'parquet_sha256': pipeline.quality.sha256_file(OUTPUT),
}
summary_path = WORKDIR / 'qwen3_14b_awq_postrun_qa_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary_md = [
    '# Qwen3-14B-AWQ (4-bit AWQ) title — QA sau full run', '',
    f"- Số dòng: {summary['rows']:,}",
    f"- Title duy nhất: {summary['unique_titles']:,}",
    f"- Trạng thái QA: `{json.dumps(status_counts, ensure_ascii=False)}`",
    f"- Title có cờ cần xem: {summary['flagged_unique_titles']:,}",
    f"- SHA-256 Parquet: `{summary['parquet_sha256']}`",
    '- `semantic_verified=False`: chưa có người xác nhận ngữ nghĩa.',
    '',
    f'- Danh sách đầy đủ: `{all_review_path.name}`',
    f'- Danh sách cần xem: `{flagged_path.name}`',
]
(WORKDIR / 'qwen3_14b_awq_postrun_qa_summary.md').write_text('\n'.join(summary_md) + '\n', encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Đóng gói kết quả/checkpoint để tải về máy; không upload HF.
import shutil

archive_base = DRIVE_ROOT / 'title_translation_cado_qwen3_14b_awq_full_v3_sampled_results'
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=WORKDIR, base_dir='.'))
print('ZIP kết quả:', archive_path)
print('Parquet:', OUTPUT)
print('QA flags:', flagged_path)
files.download(str(archive_path))
